Imports

In [1]:
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage
from dotenv import load_dotenv
import math

load_dotenv()
print("Imports successful!")

Imports successful!


Initialize Claude

In [2]:
llm = ChatAnthropic(
    model="claude-sonnet-4-6",
    temperature=0
)
print("Claude ready!")

Claude ready!


Define the tools

In [3]:
@tool
def calculator(expression: str) -> str:
    """Evaluates a basic math expression like '2 + 2' or '15 * 4'."""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def square_root(number: float) -> str:
    """Calculates the square root of a number."""
    try:
        result = math.sqrt(number)
        return f"The square root of {number} is {result}"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def power(base: float, exponent: float) -> str:
    """Raises base to the power of exponent."""
    try:
        result = math.pow(base, exponent)
        return f"{base} to the power of {exponent} is {result}"
    except Exception as e:
        return f"Error: {str(e)}"

tools = [calculator, square_root, power]
llm_with_tools = llm.bind_tools(tools)

print(f"Tools ready: {[t.name for t in tools]}")
print("Tools bound to Claude!")

Tools ready: ['calculator', 'square_root', 'power']
Tools bound to Claude!


Conversation Memory

In [4]:
class ConversationMemory:
    def __init__(self):
        self.messages = [
            SystemMessage(content="""You are a helpful math assistant. 
            You have access to calculator, square_root, and power tools.
            Always use the appropriate tool to solve math problems accurately.
            Remember the context of our conversation and refer back to 
            previous results when the user asks follow up questions.""")
        ]
    
    def add_human_message(self, content: str):
        self.messages.append(HumanMessage(content=content))
    
    def add_ai_message(self, message):
        self.messages.append(message)
    
    def add_tool_result(self, content: str, tool_call_id: str):
        self.messages.append(
            ToolMessage(content=content, tool_call_id=tool_call_id)
        )
    
    def get_messages(self):
        return self.messages
    
    def show_history(self):
        print("\n📝 Conversation History:")
        print("-" * 50)
        for msg in self.messages:
            if isinstance(msg, SystemMessage):
                print(f"🔧 System: {msg.content[:50]}...")
            elif isinstance(msg, HumanMessage):
                print(f"👤 You: {msg.content}")
            elif isinstance(msg, AIMessage):
                print(f"🤖 Agent: {msg.content}")
            elif isinstance(msg, ToolMessage):
                print(f"🔨 Tool Result: {msg.content}")
        print("-" * 50)

memory = ConversationMemory()
print("Conversation memory ready!")

Conversation memory ready!


The Agent Loop

In [5]:
def run_conversational_agent():
    memory = ConversationMemory()
    tool_map = {t.name: t for t in tools}
    
    print("🤖 Math Assistant Ready!")
    print("💡 You can ask follow up questions and I'll remember the context.")
    print("💡 Type 'history' to see our conversation so far.")
    print("💡 Type 'quit' to exit.")
    print("=" * 50)
    
    while True:
        # Get user input
        user_input = input("\n👤 You: ").strip()
        
        # Handle special commands
        if user_input.lower() == 'quit':
            print("\n👋 Goodbye! Great math session!")
            break
        
        if user_input.lower() == 'history':
            memory.show_history()
            continue
            
        if not user_input:
            print("Please enter a question.")
            continue
        
        # Add user message to memory
        memory.add_human_message(user_input)
        
        # Get response from Claude
        response = llm_with_tools.invoke(memory.get_messages())
        memory.add_ai_message(response)
        
        # Process tool calls if any
        while response.tool_calls:
            for tool_call in response.tool_calls:
                print(f"\n🔧 Using tool: {tool_call['name']}")
                print(f"   Input: {tool_call['args']}")
                
                # Execute the tool
                selected_tool = tool_map[tool_call["name"]]
                tool_result = selected_tool.invoke(tool_call["args"])
                print(f"   Result: {tool_result}")
                
                # Save tool result to memory
                memory.add_tool_result(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"]
                )
            
            # Get final response after tool execution
            response = llm_with_tools.invoke(memory.get_messages())
            memory.add_ai_message(response)
        
        print(f"\n🤖 Agent: {response.content}")

print("Agent loop ready!")

Agent loop ready!


Run the Agent

In [ ]:
run_conversational_agent() 